# LINet Training on Omni RGB-D - Google Colab

**Complete end-to-end training pipeline for Direct Mixing ResNet (LINet) on Google Colab with A100 GPU**

---

## 📋 Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime → Change runtime type → Hardware accelerator: GPU → GPU type: A100
- [ ] **Mount Google Drive:** Your code and dataset will be stored on Drive
- [ ] **Upload dataset to Drive:** `MyDrive/datasets/OmniObject3D_pretrain_256.tar.gz` (preprocessed OmniObject3D dataset)
- [ ] **Expected Runtime:** ~2-3 hours for training

---

## 🎯 What This Notebook Does:

1. ✅ Verify A100 GPU is available
2. ✅ Mount Google Drive
3. ✅ Clone your repository to local RAM (fast I/O)
4. ✅ Copy Omni RGB-D dataset to local RAM (10-20x faster than Drive)
5. ✅ Install dependencies
6. ✅ Train LINet (Direct Mixing ResNet) with all optimizations
7. ✅ Save checkpoints to Drive (persistent storage)
8. ✅ Generate training curves and analysis

---

## 🧠 About LINet:

**LINet** (Direct Mixing Network) is a 2-stream neural network architecture where:
- **RGB stream** processes color images
- **Depth stream** processes depth maps
- **Integrated Stream** combines both streams using learned scalar mixing weights at every layer

Unlike traditional fusion methods, LINet performs integration **inside each convolution neuron** through scalar-based direct mixing:
- Per-stream weights (full kernels for RGB and Depth)
- Integrated weight (1×1 channel-wise for integrated features)
- Scalar mixing coefficients (α, γ) learned per layer to combine stream outputs

This allows the network to learn optimal integration strategies at every layer with minimal computational overhead!

---

**Let's get started!** 🚀

## 1. Environment Setup & GPU Verification

In [1]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

# Check PyTorch and CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    # Check if it's A100
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\n✅ A100 GPU detected - PERFECT for training!")
    elif 'V100' in gpu_name:
        print("\n✅ V100 GPU detected - Good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\n⚠️  T4 GPU detected - Will be slower, consider upgrading to A100")
    else:
        print(f"\n⚠️  GPU: {gpu_name} - Consider using A100 for best performance")
else:
    print("\n❌ NO GPU DETECTED!")
    print("Please enable GPU: Runtime → Change runtime type → Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

GPU VERIFICATION
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU Device: NVIDIA A100-SXM4-80GB
GPU Memory: 79.25 GB

✅ A100 GPU detected - PERFECT for training!



In [2]:
# Detailed GPU info
!nvidia-smi

Mon Mar 23 19:01:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   43C    P0             80W /  400W |      48MiB /  81920MiB |      0%   E. Process |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [3]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Google Drive mounted successfully!

Drive contents:
total 3117883
-rw------- 1 root root        176 Sep 21  2019 06-lab2.gdoc
-rw------- 1 root root      21621 Sep 30  2024 113-1363667-3121001@USSR24093000064918@pre-paid.png
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (1).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (2).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (3).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final.gdoc
-rw------- 1 root root        176 Jul 11  2025 2025_Gabriel_Clinger_Contractor Agreement_BASE copy.gdoc
-rw------- 1 root root      32204 Apr 18  2022 2900 On First- Welcome Home Next Steps.docx
-rw------- 1 root root       8822 Jun 24  2017 A6.docx
-rw------- 1 root root      22204 Jan 21  2023 activity (1).xlsx
-rw------- 1 root root      22161 Ja

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [4]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"  # UPDATE THIS
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"  # Local copy for fast I/O

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

# Ensure we're in a valid directory
os.chdir('/content')
print(f"Starting in: {os.getcwd()}")

# Check if repo already exists (same session, rerunning cell)
if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"\n📁 Repo already exists: {LOCAL_REPO_PATH}")
    print(f"🔄 Pulling latest changes...")

    os.chdir(LOCAL_REPO_PATH)
    !git pull
    print("✅ Repo updated")

# Clone from GitHub (first run)
else:
    # Remove old incomplete copy if exists
    if Path(LOCAL_REPO_PATH).exists():
        print(f"\n🗑️  Removing incomplete repo copy...")
        !rm -rf {LOCAL_REPO_PATH}

    print(f"\n🔄 Cloning from GitHub...")
    print(f"   Repo: {GITHUB_REPO}")
    print(f"   Destination: {LOCAL_REPO_PATH}")

    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}

    # Verify clone succeeded
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository to {LOCAL_REPO_PATH}")

    print("✅ Repo cloned successfully")
    os.chdir(LOCAL_REPO_PATH)

# Verify repo structure
print(f"\n📂 Repository structure:")
!ls -la {LOCAL_REPO_PATH}

print(f"\n✅ Working directory: {os.getcwd()}")

REPOSITORY SETUP
Starting in: /content

📁 Repo already exists: /content/Multi-Stream-Neural-Networks
🔄 Pulling latest changes...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 13 (delta 5), reused 13 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 36.90 MiB | 5.52 MiB/s, done.
From https://github.com/clingergab/Multi-Stream-Neural-Networks
   cc312a6..31f9ee1  main       -> origin/main
Updating cc312a6..31f9ee1
Fast-forward
 .../colab_LINet3_Omni_SUN_training_full.ipynb      |  1639 +
 notebooks/colab_LINet3_Omni_training.ipynb         |  5107 ++
 notebooks/colab_LiNet3_Omni_hype_tune.ipynb        | 46308 +++++++++++++++
 notebooks/colab_LiNet3_SUN_hype_tune.ipynb         | 41177 +++++++++++++
 notebooks/colab_LiNet3_SUN_hype_tune_kfold.ipynb   | 59294 ++++++++++++++++++-
 notebooks/omni_preprocess.ipynb                    | 35274 +++++++++++
 src/models/common/model_he

## 4. Install Dependencies

In [5]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] kornia

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import ray
import kornia

print("✅ All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   ray: {ray.__version__}")
print(f"   kornia: {kornia.__version__}")


Installing dependencies...
✅ All dependencies installed!
   h5py: 3.16.0
   matplotlib: 3.10.0
   ray: 2.54.0
   kornia: 0.8.2


## 5. Copy OmniObject3D Dataset to Local Disk

**Performance Note:** Local disk I/O is ~10-20x faster than Drive!

**Dataset:** OmniObject3D pretrain dataset with RGB + Depth (per-sample .pt tensors)

In [6]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/MyDrive/datasets/OmniObject3D_Pretrain_256.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/sparse_omni_256"  # Extracted location

print("=" * 60)
print("OMNIOBJECT3D PRETRAIN DATASET SETUP (2-STREAM: RGB + DEPTH)")
print("=" * 60)

# Check if already on local disk
if Path(LOCAL_DATASET_PATH).exists() and Path(f"{LOCAL_DATASET_PATH}/class_names.txt").exists():
    print(f"Already on local disk: {LOCAL_DATASET_PATH}")
    # Count samples
    n_pt = len(list(Path(LOCAL_DATASET_PATH).rglob("*_rgb.pt")))
    print(f"   RGB-D pairs: {n_pt}")

# Copy and extract from Drive
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found on Drive: {DRIVE_DATASET_TAR}")
    print(f"Copying to local disk...")

    tar_name = Path(DRIVE_DATASET_TAR).name
    local_tar = f"/dev/shm/{tar_name}"

    !rsync -ah --info=progress2 {DRIVE_DATASET_TAR} {local_tar}

    print(f"\nExtracting dataset to local disk...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"

    !rm {local_tar}

    n_pt = len(list(Path(LOCAL_DATASET_PATH).rglob("*_rgb.pt")))
    print(f"Extracted. RGB-D pairs: {n_pt}")

else:
    print(f"Dataset not found on Drive!")
    print(f"   Expected: {DRIVE_DATASET_TAR}")
    print(f"   Run notebooks/omni_preprocess.ipynb to create it.")
    raise FileNotFoundError(f"Dataset not found at {DRIVE_DATASET_TAR}")

print("\n" + "=" * 60)
print(f"Dataset ready at: {LOCAL_DATASET_PATH}")
print("=" * 60)


OMNIOBJECT3D PRETRAIN DATASET SETUP (2-STREAM: RGB + DEPTH)
Already on local disk: /dev/shm/sparse_omni_256
   RGB-D pairs: 204606

Dataset ready at: /dev/shm/sparse_omni_256


## 6. Setup Python Path & Import LINet

In [7]:
import sys
import os

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

# Add project to Python path
project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Verify project structure
print("Project structure:")
!ls -la {project_root}/src/models/

# Import LiNet and OmniPretrain dataloader
print("\nImporting LiNet and dataloaders...")
from src.models.linear_integration.li_net3 import li_resnet18
from src.data_utils.omnipretrain_dataset import get_omnipretrain_dataloaders
from src.training.augmentation_config import AugmentationConfig


# Import Ray Tune
from ray import train, tune
from ray.tune.schedulers import ASHAScheduler

print("✅ LINet3, dataloaders, and Ray Tune imported successfully!")

Project structure:
total 52
drwxr-xr-x 12 root root 4096 Mar 23 18:52 .
drwxr-xr-x  8 root root 4096 Mar 23 18:52 ..
drwxr-xr-x  3 root root 4096 Mar 23 18:52 abstracts
drwxr-xr-x  3 root root 4096 Mar 23 19:01 common
drwxr-xr-x  3 root root 4096 Mar 23 18:52 core
drwxr-xr-x  2 root root 4096 Mar 23 18:41 direct_mixing_activation
drwxr-xr-x  2 root root 4096 Mar 23 18:41 direct_mixing_bn
drwxr-xr-x  2 root root 4096 Mar 23 18:41 direct_mixing_conv
-rw-r--r--  1 root root 1076 Mar 23 18:41 __init__.py
drwxr-xr-x  5 root root 4096 Mar 23 18:52 linear_integration
drwxr-xr-x  3 root root 4096 Mar 23 18:52 multi_channel
drwxr-xr-x  2 root root 4096 Mar 23 18:52 __pycache__
drwxr-xr-x  2 root root 4096 Mar 23 18:41 utils

Importing LiNet and dataloaders...
✅ LINet3, dataloaders, and Ray Tune imported successfully!


## 8b. Hyperparameter Tuning with Ray Tune

Perform a wide search for optimal hyperparameters using Ray Tune.
- **Parallel Trials:** Run multiple configurations simultaneously
- **Data Subset:** Use `subset_fraction` to control data usage (e.g. 0.5 = 50%)
- **Short Duration:** Train for limited epochs per trial
- **ASHA Scheduler:** Early-stop unpromising trials
- **Train/Val Split:** Stratified 90/10 split (no k-fold)

In [8]:
import os
import time

# 1. Define Paths explicitly
mps_pipe_dir = "/tmp/nvidia-mps"
mps_log_dir = "/tmp/nvidia-log"

# 2. Create the directories (CRITICAL: Daemon fails if log dir doesn't exist)
os.makedirs(mps_pipe_dir, exist_ok=True)
os.makedirs(mps_log_dir, exist_ok=True)

# 3. Set Environment Variables for the current Python process
os.environ["CUDA_MPS_PIPE_DIRECTORY"] = mps_pipe_dir
os.environ["CUDA_MPS_LOG_DIRECTORY"] = mps_log_dir
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# 4. Configure GPU and Start Daemon using the SAME environment variables
# We use f-strings to pass the python variables into the shell command
print("Setting GPU to Exclusive Process Mode...")
!nvidia-smi -i 0 -c EXCLUSIVE_PROCESS

print("Starting MPS Daemon...")
# We explicitly pass the env vars to the shell command
!export CUDA_MPS_PIPE_DIRECTORY={mps_pipe_dir} && \
 export CUDA_MPS_LOG_DIRECTORY={mps_log_dir} && \
 nvidia-cuda-mps-control -d

# 5. Verify it is running
print("Verifying Daemon Status...")
time.sleep(1) # Give it a second to start
!ps -ef | grep mps

# Check if the pipe file actually exists
if os.path.exists(os.path.join(mps_pipe_dir, "control")):
    print("✅ MPS Control Pipe found. Setup success.")
else:
    print("❌ MPS Control Pipe NOT found. Check /tmp/nvidia-log for errors.")
    # Optional: Print logs if it failed
    !cat {mps_log_dir}/control.log

Setting GPU to Exclusive Process Mode...
Compute mode is already set to EXCLUSIVE_PROCESS for GPU 00000000:00:05.0.
All done.
Starting MPS Daemon...
An instance of this daemon is already running
Verifying Daemon Status...
root       11536       1  0 18:52 ?        00:00:00 nvidia-cuda-mps-control -d
root       12547   11536  0 18:52 ?        00:00:00 nvidia-cuda-mps-server
root       15664   15347  0 19:01 ?        00:00:00 /bin/bash -c ps -ef | grep mps
root       15666   15664  0 19:01 ?        00:00:00 grep mps
✅ MPS Control Pipe found. Setup success.


In [9]:
import random
import numpy as np

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
import torch
from collections import Counter
from sklearn.model_selection import train_test_split

from src.models.linear_integration.li_net3 import li_resnet18
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler
from src.data_utils.omnipretrain_dataset import (
    OmniPretrainDataset,
    _load_class_names,
    _load_norm_stats,
    _discover_samples,
)
from src.training.augmentation_config import AugmentationConfig
from src.utils.seed import set_seed


class TrialTerminated(Exception):
    """Raised when a trial should be terminated early."""
    pass

class RayTuneReporter:
    """Callback for reporting metrics to Ray Tune during training."""

    def __init__(self):
        self.best_accuracy = 0.0
        self.best_loss = float('inf')
        self.best_train_acc = 0.0

    def on_epoch_end(self, epoch, logs):
        """Report current AND best metrics to Ray Tune."""
        if logs['val_accuracy'] > self.best_accuracy:
            self.best_accuracy = logs['val_accuracy']
            if logs['train_accuracy'] > self.best_train_acc:
                self.best_train_acc = logs['train_accuracy']

        if logs['val_loss'] < self.best_loss:
            self.best_loss = logs['val_loss']

        gap = self.best_train_acc - self.best_accuracy
        composite = self.best_accuracy - 10 * (gap**3)

        metrics = {
            "accuracy": logs['val_accuracy'],
            "loss": logs['val_loss'],
            "best_accuracy": self.best_accuracy,
            "best_loss": self.best_loss,
            "train_loss": logs['train_loss'],
            "train_accuracy": logs['train_accuracy'],
            "best_train_acc": self.best_train_acc,
            "gap": gap,
            "composite": composite,
        }

        tune.report(metrics)


def train_linet_tune(
    config,
    data_root=None,
    norm_stats=None,
    num_classes=None,
    all_samples=None,
    class_names=None,
    subset_fraction=1.0,
    seed=42,
):
    """
    Trainable function for Ray Tune — train/val split, optional subset.

    Args:
        config: Ray Tune configuration dict with hyperparameters
        data_root: Path to dataset root
        norm_stats: Normalization statistics dict
        num_classes: Number of classes
        all_samples: Pre-discovered list of (rgb_path, depth_path, label)
        class_names: List of class name strings
        subset_fraction: Fraction of data to use (e.g. 0.5 = 50%)
        seed: Random seed for reproducible trials
    """
    set_seed(seed, deterministic=False)
    g = torch.Generator().manual_seed(seed)

    # Per-trial augmentation config
    aug_config = AugmentationConfig(
        rgb_aug_prob=config.get("rgb_aug_prob"),
        rgb_aug_mag=config.get("rgb_aug_mag"),
        depth_aug_prob=config.get("depth_aug_prob"),
        depth_aug_mag=config.get("depth_aug_mag"),
    )

    # Stratified train/val split (deterministic given seed)
    all_labels = [s[2] for s in all_samples]
    train_indices, val_indices = train_test_split(
        list(range(len(all_samples))),
        test_size=0.1,
        random_state=seed,
        stratify=all_labels,
    )

    # Apply subset_fraction
    if subset_fraction < 1.0:
        n_train = int(len(train_indices) * subset_fraction)
        n_val = int(len(val_indices) * subset_fraction)
        train_perm = torch.randperm(len(train_indices), generator=g)[:n_train]
        val_perm = torch.randperm(len(val_indices), generator=g)[:n_val]
        train_indices = [train_indices[i] for i in train_perm]
        val_indices = [val_indices[i] for i in val_perm]

    train_samples = [all_samples[i] for i in train_indices]
    val_samples = [all_samples[i] for i in val_indices]

    # Create datasets
    train_dataset = OmniPretrainDataset(
        data_root=data_root,
        split='train',
        samples=train_samples,
        class_names=class_names,
        norm_stats=norm_stats,
        normalize=False,  # GPU will normalize after augmentation
        **aug_config.to_dict(),
    )
    val_dataset = OmniPretrainDataset(
        data_root=data_root,
        split='val',
        samples=val_samples,
        class_names=class_names,
        norm_stats=norm_stats,
        normalize=False,
    )

    # Stratified sampling for training
    subset_labels = [s[2] for s in train_samples]
    label_counts = Counter(subset_labels)
    num_samples = len(subset_labels)
    class_weights = {label: num_samples / count for label, count in label_counts.items()}
    sample_weights = torch.tensor(
        [class_weights[label] for label in subset_labels], dtype=torch.float32
    )

    train_sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=num_samples,
        replacement=True,
        generator=g,
    )

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        sampler=train_sampler,
        num_workers=1,
        prefetch_factor=2,
        persistent_workers=True,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=1,
        prefetch_factor=2,
        persistent_workers=False,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )

    # Create Model
    model = li_resnet18(
        num_classes=num_classes,
        stream_input_channels=[3, 1],
        dropout_p=config["dropout_p"],
        width_multiplier=0.75,
        device="cuda",
        use_amp=True,
    )

    # Create Optimizer
    optimizer = create_stream_optimizer(
        model,
        optimizer_type='adamw',
        stream_lrs=[config["lr_rgb"], config["lr_depth"]],
        stream_weight_decays=[config["wd_rgb"], config["wd_depth"]],
        shared_lr=config["lr_shared"],
        integration_weight_decay=config["wd_integrated"],
    )

    # Create Scheduler
    warmup_epochs = 5
    scheduler = setup_scheduler(
        optimizer,
        scheduler_type='cosine',
        eta_min=[config['s1_eta_min'], config['s2_eta_min'], config['eta_min'], config['eta_min']],
        t_max=config['t_max'],
        train_loader_len=len(train_loader),
        warmup_epochs=warmup_epochs,
        warmup_start_factor=0.2,
    )

    # Compile model
    model.compile(
        optimizer=optimizer,
        scheduler=scheduler,
        loss='cross_entropy',
        label_smoothing=config["label_smoothing"],
        gpu_augmentation=True,
        norm_stats=norm_stats,
        **aug_config.to_dict(),
    )

    # Train
    try:
        model.fit(
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=100,
            early_stopping=True,
            patience=15,
            grad_clip_norm=config["grad_clip_norm"],
            modality_dropout=True,
            modality_dropout_start=config['modality_dropout_start'],
            modality_dropout_ramp=config['modality_dropout_ramp'],
            modality_dropout_rate=config['modality_dropout_rate'],
            callbacks=[RayTuneReporter()],
            verbose=False,
        )
    except TrialTerminated as e:
        print(f"\n{e}")


In [10]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# Ray Tune saves ALL experiment state to DRIVE_STORAGE_PATH via storage_path.
# When Colab dies, re-run the notebook — Tuner.restore() picks up where it
# left off. Completed trials preserved, interrupted trials restart.
# =============================================================================

import hashlib
import json as json_module
import os
import pandas as pd
from pathlib import Path

# --- Ray Tune persistent storage on Google Drive ---
DRIVE_STORAGE_PATH = "/content/drive/MyDrive/ray_tune_experiments"
EXPERIMENT_NAME = "omni_pretrain_hpo"

# --- subset_fraction: fraction of train/val data to use per trial ---
subset_fraction = 1

SEED = 42
NUM_SAMPLES = 50  # Total trials to run across all sessions


Path(DRIVE_STORAGE_PATH).mkdir(parents=True, exist_ok=True)

experiment_path = os.path.join(DRIVE_STORAGE_PATH, EXPERIMENT_NAME)
RESUME_EXISTING = os.path.exists(experiment_path)

print(f"Drive storage: {DRIVE_STORAGE_PATH}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Resume existing: {RESUME_EXISTING}")
print(f"Subset fraction: {subset_fraction}")
print(f"Total trials: {NUM_SAMPLES}")
if RESUME_EXISTING:
    print(f"\n  Previous experiment found at {experiment_path}")
    print(f"  Tuner.restore() will resume where the last session left off.")


Warm-start: ENABLED
Subset fraction: 1
   Results CSV: /content/drive/MyDrive/ray_tune_results/omni_pretrain_results.csv
   Epoch history: /content/drive/MyDrive/ray_tune_results/omni_pretrain_epoch_history.csv
   HyperOpt checkpoint dir: /content/drive/MyDrive/ray_tune_results


In [ ]:
# Initialize Ray
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune.search import ConcurrencyLimiter
from ray.tune import CLIReporter

os.environ["RAY_AIR_NEW_OUTPUT"] = "0"  # must be set BEFORE ray.init()

ray.shutdown()
ray.init(
    ignore_reinit_error=True,
    runtime_env={
        "env_vars": {
            "CUDA_MPS_PIPE_DIRECTORY": "/tmp/nvidia-mps",
            "CUDA_MPS_LOG_DIRECTORY": "/tmp/nvidia-log",
            "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
            "CUDA_VISIBLE_DEVICES": "0",
        }
    }
)

# Load dataset metadata (once, shared across all trials)
class_names = _load_class_names(LOCAL_DATASET_PATH)
norm_stats = _load_norm_stats(LOCAL_DATASET_PATH)
all_samples = _discover_samples(LOCAL_DATASET_PATH, class_names)
num_classes = len(class_names)

print(f"Dataset: {LOCAL_DATASET_PATH}")
print(f"  Classes: {num_classes}")
print(f"  Total samples: {len(all_samples)}")
print(f"  Subset fraction: {subset_fraction}")


# Define trainable (same for both new and restored runs)
trainable = tune.with_resources(
    tune.with_parameters(
        train_linet_tune,
        data_root=LOCAL_DATASET_PATH,
        norm_stats=norm_stats,
        num_classes=num_classes,
        all_samples=all_samples,
        class_names=class_names,
        subset_fraction=subset_fraction,
        seed=SEED,
    ),
    resources={"cpu": 1, "gpu": 0.1},
)


if RESUME_EXISTING:
    # =========================================================
    # RESUME: Restore previous experiment from Google Drive
    # =========================================================
    # Restores: completed trials, ASHA scheduler state, HyperOpt
    # search algorithm state. Interrupted trials restart from epoch 0.
    print("\n" + "=" * 60)
    print("RESUMING EXPERIMENT FROM GOOGLE DRIVE")
    print("=" * 60)

    tuner = tune.Tuner.restore(
        path=experiment_path,
        trainable=trainable,
        resume_unfinished=True,
        resume_errored=True,
    )

else:
    # =========================================================
    # NEW: Create fresh experiment
    # =========================================================
    print("\n" + "=" * 60)
    print("STARTING NEW EXPERIMENT")
    print("=" * 60)

    # Search space
    search_space = {
        # Learning rates
        "lr_rgb": tune.loguniform(1e-5, 5e-4),
        "lr_depth": tune.loguniform(1e-5, 5e-4),
        "lr_shared": tune.loguniform(5e-6, 1e-4),

        # Weight decay
        "wd_rgb": tune.loguniform(1e-6, 5e-4),
        "wd_depth": tune.loguniform(1e-6, 5e-4),
        "wd_integrated": tune.loguniform(1e-5, 1e-3),

        # Scheduler eta_min
        "s1_eta_min": tune.uniform(5e-8, 5e-6),
        "s2_eta_min": tune.uniform(5e-8, 5e-6),
        "eta_min": tune.uniform(1e-8, 1e-6),

        # Scheduler t_max / batch size
        "t_max": tune.choice([95]),
        "batch_size": tune.choice([96, 128]),

        # Regularization
        "dropout_p": tune.uniform(0.35, 0.55),
        "label_smoothing": tune.uniform(0.001, 0.15),
        "grad_clip_norm": tune.uniform(0.5, 1.5),

        # Augmentation parameters
        "rgb_aug_prob": tune.uniform(0.9, 1.8),
        "rgb_aug_mag": tune.uniform(0.9, 1.8),
        "depth_aug_prob": tune.uniform(0.9, 1.8),
        "depth_aug_mag": tune.uniform(0.9, 1.8),

        # Modality dropout
        "modality_dropout_rate": tune.uniform(0.12, 0.17),
        "modality_dropout_start": tune.choice([0]),
        "modality_dropout_ramp": tune.choice([20]),
    }

    hyperopt_searcher = HyperOptSearch(
        metric="composite",
        mode="max",
        random_state_seed=SEED,
    )

    limited_search_alg = ConcurrencyLimiter(
        hyperopt_searcher,
        max_concurrent=10,
    )

    reporter = CLIReporter(
        parameter_columns=[
            "lr_rgb", "lr_depth", "lr_shared",
            "wd_rgb", "wd_depth", "wd_integrated",
            "batch_size", "dropout_p",
            "label_smoothing", "grad_clip_norm",
            "rgb_aug_prob", "rgb_aug_mag",
            "depth_aug_prob", "depth_aug_mag",
            "modality_dropout_rate", "modality_dropout_start", "modality_dropout_ramp",
        ],
        metric_columns={
            "training_iteration": "iter",
            "best_accuracy": "best_accuracy",
            "best_train_acc": "best_train_acc",
            "composite": "composite",
        },
        max_report_frequency=30,
        print_intermediate_tables=True,
    )

    asha_scheduler = ASHAScheduler(
        time_attr="training_iteration",
        metric="accuracy",
        mode="max",
        max_t=100,
        grace_period=5,
        reduction_factor=2,
    )

    tuner = tune.Tuner(
        trainable,
        param_space=search_space,
        tune_config=tune.TuneConfig(
            scheduler=asha_scheduler,
            search_alg=limited_search_alg,
            num_samples=NUM_SAMPLES,
        ),
        run_config=ray.tune.RunConfig(
            storage_path=DRIVE_STORAGE_PATH,
            name=EXPERIMENT_NAME,
            progress_reporter=reporter,
            verbose=1,
        ),
    )


# Run (or resume) tuning
print("\n" + "=" * 60)
print("STARTING HYPERPARAMETER TUNING")
print("=" * 60)

results = tuner.fit()

best_result = results.get_best_result("best_accuracy", "max")

print("\n" + "=" * 60)
print("TUNING COMPLETE")
print("=" * 60)
print(f"Best Trial Config: {best_result.config}")
print(f"Best Trial Accuracy: {best_result.metrics['best_accuracy']:.4f}")
print(f"Best Trial Loss: {best_result.metrics['best_loss']:.4f}")
print(f"\nExperiment saved to: {experiment_path}")
print(f"To resume after Colab dies: just re-run this notebook.")


2026-03-23 19:01:52,540	INFO worker.py:2013 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Streaming output truncated to the last 5000 lines.
| train_linet_tune_efb9aece | RUNNING  | 172.28.0.12:17079 | 0.000163816 | 0.00029524  | 8.08837e-05 | 1.93944e-05 | 4.80092e-06 |     8.31683e-05 |          128 |    0.481934 |         0.139999  |         0.691598 |       1.47736  |      1.49768  |         1.62231  |        1.08982  |               0.127709 |                      0 |                     20 |      1 |       0.197204  |        0.0534905 |   0.226887  |
| train_linet_tune_93ddcc7f | RUNNING  | 172.28.0.12:17191 | 0.000273625 | 0.000145514 | 2.2519e-05  | 5.23178e-05 | 3.38082e-06 |     0.000102576 |          128 |    0.417339 |         0.0028015 |         1.21804  |       1.48898  |      1.77688  |         1.09883  |        1.56178  |               0.166071 |                      0 |                     20 |      1 |       0.0673476 |        0.0272014 |   0.0679947 |
| train_linet_tune_c7a8ef5d | RUNNING  | 172.28.0.12:17311 | 1.7525e-05  | 2.41149e-05 | 4.1632e-05  | 8.

In [ ]:
# =============================================================================
# SAVE RESULTS CSV (for offline analysis)
# =============================================================================
# Ray Tune already saved everything to DRIVE_STORAGE_PATH.
# This cell just exports a clean CSV for easy analysis.
# =============================================================================

import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

results_df = results.get_dataframe()

csv_dir = f"{DRIVE_STORAGE_PATH}/analysis"
Path(csv_dir).mkdir(parents=True, exist_ok=True)

csv_path = f"{csv_dir}/omni_hpo_results_{timestamp}.csv"
results_df.to_csv(csv_path, index=False)
print(f"Results CSV saved: {csv_path}")
print(f"  Trials: {len(results_df)}")

# Also save a latest copy
latest_path = f"{csv_dir}/omni_hpo_results_latest.csv"
results_df.to_csv(latest_path, index=False)
print(f"Latest copy: {latest_path}")


In [ ]:
# Analyze Top 10 Trials from Ray Tune (ranked by best_accuracy)
import pandas as pd

print("=" * 80)
print("TOP 10 TRIALS BY BEST ACCURACY (CONTINUOUS SEARCH)")
print("=" * 80)

df = results.get_dataframe()

df_sorted = df.sort_values('best_accuracy', ascending=False)

display_cols = [
    'best_accuracy', 'best_train_acc', 'composite',
    'config/lr_rgb', 'config/lr_depth', 'config/lr_shared',
    'config/wd_rgb', 'config/wd_depth', 'config/wd_integrated',
    'config/s1_eta_min', 'config/s2_eta_min', 'config/eta_min', 'config/t_max',
    'config/dropout_p', 'config/label_smoothing', 'config/grad_clip_norm',
    'config/rgb_aug_prob', 'config/rgb_aug_mag',
    'config/depth_aug_prob', 'config/depth_aug_mag',
    'config/modality_dropout_rate', 'config/modality_dropout_start', 'config/modality_dropout_ramp',
]

top_10 = df_sorted[display_cols].head(10)

top_10_formatted = top_10.copy()
top_10_formatted['best_accuracy'] = top_10_formatted['best_accuracy'].apply(lambda x: f"{x*100:.2f}%")

sci_cols = [
    'config/lr_rgb', 'config/lr_depth', 'config/lr_shared',
    'config/wd_rgb', 'config/wd_depth', 'config/wd_integrated',
    'config/s1_eta_min', 'config/s2_eta_min', 'config/eta_min',
]
for col in sci_cols:
    if col in top_10_formatted.columns:
        top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.2e}")

float_cols = [
    'config/dropout_p', 'config/label_smoothing', 'config/grad_clip_norm',
    'config/rgb_aug_prob', 'config/rgb_aug_mag',
    'config/depth_aug_prob', 'config/depth_aug_mag',
    'config/modality_dropout_rate',
]
for col in float_cols:
    if col in top_10_formatted.columns:
        top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.3f}")

print(top_10_formatted.to_string(index=False))
print("\n" + "=" * 80)


In [ ]:
# =============================================================================
# ANALYZE TOP 10 TRIALS BY COMPOSITE
# =============================================================================

import pandas as pd

RESULTS_CSV_PATH = f"{DRIVE_STORAGE_PATH}/analysis/omni_hpo_results_latest.csv"
TARGET_HASH = "PASTE_HASH_HERE"  # <-- paste your search_space_hash

df = pd.read_csv(RESULTS_CSV_PATH)

df["search_space_hash"] = (
    df["search_space_hash"].astype(str).str.replace(r"\.0$", "", regex=True)
)
target = str(TARGET_HASH).strip()
df = df[df["search_space_hash"] == target]
print(f"Trials matching hash {target}: {len(df)}")

df = df.sort_values("composite", ascending=False)
top_10 = df.head(10).copy()

config_cols = [c for c in df.columns if c.startswith("config/")]

print("\n" + "=" * 80)
print(f"TOP 10 TRIALS BY COMPOSITE (hash: {target})")
print("=" * 80)

for rank, (_, row) in enumerate(top_10.iterrows(), 1):
    gap = row["best_train_acc"] - row["best_accuracy"]
    print(f"\n--- #{rank} | Composite: {row['composite']*100:.2f}% | "
          f"Accuracy: {row['best_accuracy']*100:.2f}% | "
          f"Gap: {gap*100:.1f}pp ---")

print("\n" + "=" * 80)
print("HYPERPARAMETER RANGES ACROSS TOP 10 TRIALS")
print("=" * 80)
print(f"{'Parameter':<35} {'Min':>12} {'Max':>12} {'Median':>12}")
print("-" * 75)

for col in config_cols:
    short_name = col.replace("config/", "")
    col_min = top_10[col].min()
    col_max = top_10[col].max()
    col_med = top_10[col].median()
    if abs(col_med) < 0.001:
        print(f"{short_name:<35} {col_min:>12.2e} {col_max:>12.2e} {col_med:>12.2e}")
    else:
        print(f"{short_name:<35} {col_min:>12.4f} {col_max:>12.4f} {col_med:>12.4f}")

print("\n" + "=" * 80)
print("FULL CONFIG TABLE")
print("=" * 80)

top_10["gap"] = top_10["best_train_acc"] - top_10["best_accuracy"]

display_df = top_10[["composite", "best_accuracy", "best_train_acc", "gap", "training_iteration"] + config_cols].copy()
display_df.insert(0, "rank", range(1, len(display_df) + 1))
display_df["best_accuracy"] = display_df["best_accuracy"].apply(lambda x: f"{x*100:.2f}%")
display_df["training_iteration"] = display_df["training_iteration"].astype(int)

sci_cols = [c for c in config_cols if any(k in c for k in ["lr_", "wd_", "eta_min"])]
float_cols = [c for c in config_cols if c not in sci_cols and c != "config/t_max"]

for col in sci_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.2e}")
for col in float_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(
            lambda x: f"{int(x)}" if col == "config/t_max" else f"{x:.3f}"
        )

display_df.columns = [c.replace("config/", "") for c in display_df.columns]

for col in ["wd_shared", "optimizer_type", "scheduler_type", "epochs"]:
    if col in display_df.columns:
        del display_df[col]

cols = list(display_df.columns)
if "wd_integrated" in cols and "wd_depth" in cols:
    cols.remove("wd_integrated")
    cols.insert(cols.index("wd_depth") + 1, "wd_integrated")
    display_df = display_df[cols]

print(display_df.to_string(index=False))
